# Tools for Forecasting Rat Sightings

Here, we should write down some detailed instructions. Each section follows a similar approach. 

### Instructions for Rat Sightings: 14 Day Forecasts

For this section, the purpose is make a 14 day forecast of rat sightings in New York City as a whole. The model will make a prediction based on weather data and [data on rat sightings](https://data.cityofnewyork.us/Social-Services/Rat-Sightings/3q43-55fe/about_data) up to the last time the rat sightings dataset was updated. To use, one simply runs all the code blocks in the given section. Here is how it works.

0. One should first set the parameters to be used for the Neural Prophet model. There are some baseline parameters which were obtained by using Optuna. One should change these parameters based on need. Tuning hyperparameters can take quite some time and so we recommend only doing so if the performance of the model is being affected by suboptimal choices of parameters. We will address how to do this below. *To-do: write the Optuna hyperparameter tuning to be done to optimize these parameters once new data is obtained.*

1. Next, it imports needed to make the forecasts. It then download the data on Rat Sightings from [NYC Open Data's Rat Sightings Dataset](https://data.cityofnewyork.us/Social-Services/Rat-Sightings/3q43-55fe/about_data) and also downloads the weather data up to the last day which appears in the Rat Sightings data. For a 14 day forecast, it will assume that the weather on the last day will remain the same for the next 14 days. 

3. Using the weather data and Rat Sightings data, it then trains a Neural Prophet model. **Need to explain how the model works.**

4. Using the trained model, it then forecasts 14 days out. 



### Warnings

## Rat Sightings: 14 Day Forecasts

### Set Parameters

In [ ]:
parameters = dict()
parameters['apparent_temperature_max'] = 30
parameters['apparent_temperature_min'] = 5
parameters['snowfall_sum'] = 1
parameters['n_lags'] = 16
parameters['epochs'] = 60
parameters['learning_rate'] = 0.2986532324899507 
parameters['batch_size'] = 128
parameters['ar_reg'] = 2.132925719127823



parameters['n_forecasts'] = 14 

params = parameters.copy()

{'lag_temp_max': 30, 'lag_temp_min': 5, 'lag_snowfall': 1, 'n_lags': 16, 'epochs': 60, 'learning_rate': 0.2986532324899507, 'batch_size': 128 'ar_reg': 2.132925719127823}

### Import Packages

In [ ]:
import requests 
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression

from prophet import Prophet
from pandas.tseries.holiday import USFederalHolidayCalendar
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from prophet.plot import add_changepoints_to_plot
from prophet.plot import plot_plotly, plot_components_plotly
import itertools

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)

import optuna
from neuralprophet import NeuralProphet
import logging



In [ ]:
# We downloads the rat sightings data to a folder.

import requests

url = "https://data.cityofnewyork.us/api/views/3q43-55fe/rows.csv?accessType=DOWNLOAD"

response = requests.get(url)
response.raise_for_status()

with open("../scr/data/rat_sightings_data/Rat_Sightings_NYC.csv", "wb") as f:
    f.write(response.content)

In [ ]:
rat_sighting = pd.read_csv("../scr/data/rat_sightings_data/Rat_Sightings_NYC.csv")


In [ ]:
rat_sighting.columns = [t.partition('(')[0].strip().lower().replace(' ', '_') for t in rat_sighting.columns] #apply to column headers
rat_sighting['location_type'] = rat_sighting['location_type'].str.strip().str.replace(' ', '_').str.lower()  #apply to location_type column
cols_to_drop = [c for c in rat_sighting.columns if (rat_sighting[c].nunique(dropna=False) == 1)]
rat_sighting = rat_sighting.drop(columns=cols_to_drop)
rat_sighting['created_date'] = pd.to_datetime(rat_sighting['created_date']) 
rat_sighting = rat_sighting.drop(columns='park_borough')
rat_sighting = rat_sighting.drop(columns=['location'])
rs = rat_sighting.copy()
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs = rs[rs['created_date']>='2020-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

The forecasted values are the naive forecast i.e. we take the last observed day and assume that these are good predictions for the next 14 days.display_svg
    
We really should find a way to get forecast data that's better though.

In [ ]:
lat, lon = 40.7831, -73.9712
last_date = rs['ds'].max()
start = "2020-01-01"
end   = last_date.strftime("%Y-%m-%d")  # use last date of rs

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd['ds'] = pd.to_datetime(nd['date'])
    wd = nd.drop(columns=['date'])
else:
    wd = pd.DataFrame(data["daily"])
    wd["ds"] = pd.to_datetime(wd["time"])
    wd = wd.drop(columns=["time"])

wd = wd.reset_index(drop=True)
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1),
                             periods=14,
                             freq='D')

last_row = wd.iloc[-1]
wd_14 = pd.DataFrame([last_row.values] * 14, columns=wd.columns)
wd_14['ds'] = future_dates 
wd_14 = wd_14.reset_index(drop=True)

In [ ]:
# Suppress cmdstanpy info logs
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']

wd["ds"] = pd.to_datetime(wd["ds"])
wd_14["ds"] = pd.to_datetime(wd_14["ds"])

rs["ds"] = pd.to_datetime(rs["ds"])

rs = rs.merge(wd[['ds'] + regressed_features], on="ds", how="left")

### Fixing the Seed

We need to fix the seed for reproducibility. Without the following code block, the forecasts and *will* change on each run.

In [ ]:
import random
import torch

# set seeds for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# make PyTorch deterministic
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

### Train the model

In [ ]:
from neuralprophet import NeuralProphet
import pandas as pd

forecast_horizon = 14
regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']

model = NeuralProphet(yearly_seasonality=True, 
                      weekly_seasonality=True, 
                      epochs=params['epochs'], 
                      learning_rate=params['learning_rate'], 
                      batch_size=params['batch_size'], 
                      n_forecasts = params['n_forecasts'], 
                      n_lags = params['n_lags'], 
                      ar_reg = params['ar_reg'], 
                      )
model = model.add_country_holidays(country_name="US")

for col in regressed_features:
    model.add_lagged_regressor(col, n_lags=params[col])

model.fit(rs, freq="D")
future = model.make_future_dataframe(rs, periods=forecast_horizon, n_historic_predictions=28)

for col in regressed_features:
    future[col] = pd.concat([rs[col], wd_14[col]], ignore_index=True)

forecast = model.predict(future)

rs_df = model.get_latest_forecast(forecast, include_previous_forecasts = 14)

### Forecast for 14 Days

In [ ]:
df_wide= rs_df.copy()

# use the last 14 entries of 'origin-0' as the forecast
forecast_horizon = 14
origin_col = 'origin-0' 
last_14_preds = df_wide[origin_col].dropna().tail(forecast_horizon)

# compute corresponding forecast dates
last_14_dates = pd.to_datetime(df_wide['ds'].tail(forecast_horizon))

forecast_vertical = pd.DataFrame({'ds': last_14_dates, 'yhat': last_14_preds.values})

for _, row in forecast_vertical.iterrows():
    print(f"On day {row['ds'].date()}, the model predicts {round(row['yhat'])} rat sightings reported to 311.")